# Arrays and Collections

For multi-dimensional data (e.g., scenario × time, or region × quantile × time) use `TimeSeriesArray`. For grouping heterogeneous time series that don't share the same timestamps, use `TimeSeriesCollection`.

## TimeSeriesArray

An array stores an N-dimensional array with named `Dimension` objects. Each dimension has a name and a list of labels (datetimes, strings, or numbers).  
Common use cases include ensemble forecasts, scenario analysis, and multi-site probabilistic forecasts.

In [ ]:
from datetime import datetime, timedelta, timezone

import numpy as np

import timedatamodel as tdm

rng = np.random.default_rng(42)
base = datetime(2024, 1, 15, tzinfo=timezone.utc)

### Building a 4D array

Imagine a wind power forecasting system that produces:

- **3 knowledge times** — forecasts issued at 00:00, 06:00, and 12:00
- **24 valid times** — hourly forecast horizon
- **3 wind farms** — Alpha, Bravo, Charlie
- **5 quantiles** — probabilistic spread from q10 to q90

That gives a `(3, 24, 3, 5)` array with **1 080 values**.

In [ ]:
knowledge_times = [base + timedelta(hours=h) for h in [0, 6, 12]]
valid_times = [base + timedelta(hours=h) for h in range(24)]
wind_farms = ["Alpha", "Bravo", "Charlie"]
quantiles = ["q10", "q25", "q50", "q75", "q90"]

nk = len(knowledge_times)
nv = len(valid_times)
nw = len(wind_farms)
nq = len(quantiles)

data = np.empty((nk, nv, nw, nq))
for k in range(nk):
    for w in range(nw):
        capacity = 50 + 20 * w
        daily_shape = capacity * (1 + 0.3 * np.sin(np.linspace(0, 2 * np.pi, nv)))
        for q_idx, q_spread in enumerate([-0.4, -0.15, 0, 0.15, 0.4]):
            data[k, :, w, q_idx] = daily_shape * (1 + q_spread) + rng.normal(0, 3, nv) + k * 5

cube = tdm.TimeSeriesArray(
    tdm.Frequency.PT1H,
    timezone="UTC",
    name="wind_power",
    unit="MW",
    data_type=tdm.DataType.FORECAST,
    dimensions=[
        tdm.Dimension("knowledge_time", knowledge_times),
        tdm.Dimension("valid_time", valid_times),
        tdm.Dimension("wind_farm", wind_farms),
        tdm.Dimension("quantile", quantiles),
    ],
    values=data,
)
cube

### Array properties

In [ ]:
print(f"Shape:      {cube.shape}")
print(f"Dimensions: {cube.dim_names}")
print(f"N dims:     {cube.ndim}")
print(f"Begin:      {cube.begin}")
print(f"End:        {cube.end}")
print(f"Has missing:{cube.has_missing}")

In [ ]:
coords = cube.coords
for dim_name, labels in coords.items():
    preview = labels[:3]
    suffix = f" ... ({len(labels)} total)" if len(labels) > 3 else ""
    print(f"{dim_name:18s} {preview}{suffix}")

### `sel()` — fixing one dimension (4D → 3D array)

Fix `knowledge_time` to get all farms and quantiles for a single forecast issuance.  
The result is still an array because 3 dimensions remain.

In [ ]:
noon_forecast = cube.sel(knowledge_time=knowledge_times[2])  # 12:00 issuance

print(f"Type:  {type(noon_forecast).__name__}")
print(f"Shape: {noon_forecast.shape}")
print(f"Dims:  {noon_forecast.dim_names}")

### `sel()` — fixing two dimensions (4D → 2D → TimeSeriesTable)

Fix `knowledge_time` and `wind_farm` to see all quantiles over time for one farm.  
Only 2 dimensions remain (`valid_time × quantile`), so the array auto-collapses to a `TimeSeriesTable`.

In [ ]:
alpha_quantiles = cube.sel(
    knowledge_time=knowledge_times[2],
    wind_farm="Alpha",
)

print(f"Type:    {type(alpha_quantiles).__name__}")
print(f"Columns: {alpha_quantiles.column_names}")
alpha_quantiles

Fix a different pair — `knowledge_time` and `quantile` — to compare farms at the median:

In [ ]:
farms_median = cube.sel(
    knowledge_time=knowledge_times[2],
    quantile="q50",
)

print(f"Type:    {type(farms_median).__name__}")
print(f"Columns: {farms_median.column_names}")
farms_median

### `sel()` — fixing three dimensions (4D → 1D → TimeSeries)

Fix everything except `valid_time` to extract a single time series.

In [ ]:
single = cube.sel(
    knowledge_time=knowledge_times[2],
    wind_farm="Alpha",
    quantile="q50",
)

print(f"Type: {type(single).__name__}")
print(f"Len:  {len(single)}")
single

### `isel()` — index-based selection

Use integer positions instead of labels.

In [ ]:
bravo_p90 = cube.isel(
    knowledge_time=0,
    wind_farm=1,       # Bravo
    quantile=-1,       # q90 (last)
)

print(f"Type: {type(bravo_p90).__name__}, len={len(bravo_p90)}")
print(f"Mean: {np.nanmean(bravo_p90.arr):.1f} MW")

### Slicing a dimension

Use `slice()` to keep a range of labels. The dimension is preserved (not collapsed).

In [ ]:
narrow = cube.sel(
    knowledge_time=knowledge_times[2],
    wind_farm="Alpha",
    quantile=slice("q25", "q75"),
)

print(f"Type:    {type(narrow).__name__}")
print(f"Columns: {narrow.column_names}")
narrow

### Converting to pandas

`to_pandas_dataframe()` produces a long-format DataFrame with a `MultiIndex` covering all dimensions.

In [ ]:
df = cube.to_pandas_dataframe()
print(f"Shape:        {df.shape}")
print(f"Index levels: {list(df.index.names)}")
df.head(10)

### Building an array from a list of TimeSeries

`from_timeseries_list()` is handy when you already have individual forecasts.

In [ ]:
base_prices = 50 + 20 * np.sin(np.linspace(0, 2 * np.pi, 24))

series_list = [
    tdm.TimeSeries(
        tdm.Frequency.PT1H,
        timestamps=valid_times,
        values=(base_prices * factor + rng.normal(0, 2, 24)).tolist(),
        name="price",
        unit="EUR/MWh",
    )
    for factor in [0.7, 0.85, 1.0, 1.15, 1.3]
]

ensemble = tdm.TimeSeriesArray.from_timeseries_list(
    series_list,
    dimension=tdm.Dimension("percentile", ["p10", "p25", "p50", "p75", "p90"]),
    name="price_ensemble",
)
ensemble

---

## TimeSeriesCollection

A `TimeSeriesCollection` groups time series that may have different frequencies, time ranges, or numbers of points. Think of it as a named bag of `TimeSeries` and `TimeSeriesTable` objects.

In [ ]:
daily_base = datetime(2024, 1, 1, tzinfo=timezone.utc)
hours = [base + timedelta(hours=i) for i in range(24)]

ts_hourly = tdm.TimeSeries(
    tdm.Frequency.PT1H,
    timestamps=hours,
    values=[100.0 + rng.normal(0, 10) for _ in range(24)],
    name="wind_hourly",
    unit="MW",
)

ts_daily = tdm.TimeSeries(
    tdm.Frequency.P1D,
    timestamps=[daily_base + timedelta(days=d) for d in range(30)],
    values=[2400.0 + rng.normal(0, 200) for _ in range(30)],
    name="wind_daily_energy",
    unit="MWh",
)

ts_15min = tdm.TimeSeries(
    tdm.Frequency.PT15M,
    timestamps=[base + timedelta(minutes=15 * i) for i in range(96)],
    values=[50.0 + rng.normal(0, 5) for _ in range(96)],
    name="solar_15min",
    unit="MW",
)

### Creating a collection

In [ ]:
collection = tdm.TimeSeriesCollection(
    [ts_hourly, ts_daily, ts_15min],
    name="Plant overview",
    description="Mixed-frequency data for a single plant",
)
collection

### Dictionary-like access

In [ ]:
print(f"Names: {collection.names}")
print(f"Count: {collection.series_count}")

collection["wind_hourly"]

### Adding and removing series

Collections are immutable — `add()` and `remove()` return new collections.

In [ ]:
ts_price = tdm.TimeSeries(
    tdm.Frequency.PT1H,
    timestamps=hours,
    values=[45.0 + rng.normal(0, 8) for _ in range(24)],
    name="spot_price",
    unit="EUR/MWh",
)

extended = collection.add(ts_price)
print(f"Original: {collection.names}")
print(f"Extended: {extended.names}")

In [ ]:
reduced = extended.remove("wind_daily_energy")
print(f"Reduced: {reduced.names}")

### Iterating over a collection

In [ ]:
for name, series in collection.items():
    print(f"{name:20s}  freq={str(series.frequency):5s}  len={len(series):3d}  begin={series.begin}")

## Summary

- **`TimeSeriesArray`**: N-dimensional time series with `Dimension` labels; slice with `sel()` / `isel()`; auto-collapses to Table or Series
- **`TimeSeriesCollection`**: heterogeneous container for series with different frequencies and time ranges; dictionary-like access; immutable add/remove

Next up: **nb_07** covers data quality tools — coverage bars and validation.